In [ ]:
import pandas as pd
import numpy as np
import re
import pickle
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import string
from sklearn.utils import class_weight

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

In [ ]:

df = pd.read_csv("balanced.csv") 
df.head()

In [ ]:
df.info()

In [ ]:
df['fraudulent'].value_counts()

In [ ]:
df['text'] = df['title'].fillna('') + " " + \
             df['company_profile'].fillna('') + " " + \
             df['description'].fillna('') + " " + \
             df['requirements'].fillna('')
df['text']

In [ ]:
df = df[['text','fraudulent']]
df.head()
df.shape

In [ ]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'http\S+', '', text)

    text = re.sub(r'\d+', '', text)

    text = text.translate(str.maketrans('', '', string.punctuation))

    text = text.strip()

    return text
df['text'] = df['text'].apply(clean_text)

In [ ]:
X = df['text']
y = df['fraudulent']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
max_words = 20000

tokenizer = Tokenizer(num_words=max_words)

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_seq

In [ ]:
max_len = 200

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

In [ ]:
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))

model.add(Bidirectional(LSTM(64)))

model.add(Dropout(0.5))

model.add(Dense(32, activation='relu'))

model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print(class_weights)

In [ ]:
history=model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test_pad, y_test),
    class_weight=class_weights
)

In [ ]:
pred = model.predict(X_test_pad)

pred = (pred > 0.5).astype(int)

print(classification_report(y_test, pred))

In [ ]:
model.save("fake_job_model.h5")

In [ ]:
import pickle

with open("tokenizer.pkl","wb") as f:
    pickle.dump(tokenizer,f)